In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_6593122dbc253272a6ebfca72103e2ab'

In [ ]:
!kaggle datasets list

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content

In [ ]:
!mkdir -p /content/drive/MyDrive/medical_project

In [ ]:
%cd /content/drive/MyDrive/medical_project

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='/content/drive/MyDrive/medical_model_checkpoint.keras',
    save_best_only=True,
    monitor='val_loss')

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

In [ ]:
import shutil

# Remove the destination directory if it exists to ensure a clean extraction
if os.path.exists('chest_xray'):
    shutil.rmtree('chest_xray')

!unzip -q chest-xray-pneumonia.zip -d chest_xray

In [ ]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset

In [ ]:
# Download dataset
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p '/content/brain_mri' --unzip

In [ ]:
# Verify Training is there
!ls '/content/brain_mri'
!ls '/content/brain_mri/Training'

In [ ]:
# Save to Drive immediately so this never happens again
!cp -r '/content/brain_mri/Training' '/content/drive/MyDrive/medical_project/brain_mri/Training'

In [ ]:
import shutil

# Remove the destination directory if it exists to ensure a clean extraction
if os.path.exists('brain_mri'):
    shutil.rmtree('brain_mri')

!unzip -q brain-tumor-mri-dataset.zip -d brain_mri

In [ ]:
import os

base_path = "/content/drive/MyDrive/medical_project/medical_dataset"

folders = [
    "pneumonia/NORMAL",
    "pneumonia/PNEUMONIA",

    "brain_tumor/glioma",
    "brain_tumor/meningioma",
    "brain_tumor/pituitary",
    "brain_tumor/notumor"
]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Unified dataset folders created successfully")

In [ ]:
import shutil
from glob import glob

# Source folders
normal_paths = glob('/content/drive/MyDrive/medical_project/chest_xray/chest_xray/train/NORMAL/*')
pneumonia_paths = glob('/content/drive/MyDrive/medical_project/chest_xray/chest_xray/train/PNEUMONIA/*')

# Destination folders
normal_dest = '/content/drive/MyDrive/medical_project/medical_dataset/pneumonia/NORMAL'
pneumonia_dest = '/content/drive/MyDrive/medical_project/medical_dataset/pneumonia/PNEUMONIA'

# Copy NORMAL images
for path in normal_paths:
    shutil.copy(path, normal_dest)

# Copy PNEUMONIA images
for path in pneumonia_paths:
    shutil.copy(path, pneumonia_dest)

print("Pneumonia dataset merged successfully")

In [ ]:
import shutil
from glob import glob

classes = ['glioma', 'meningioma', 'pituitary', 'notumor']

for cls in classes:

    paths = glob(f'/content/drive/MyDrive/medical_project/brain_mri/Training/{cls}/*')

    destination = f'/content/drive/MyDrive/medical_project/medical_dataset/brain_tumor/{cls}'

    for path in paths:
        shutil.copy(path, destination)

print("Brain MRI dataset merged successfully")

In [ ]:
import os

base = "/content/drive/MyDrive/medical_project/medical_dataset"

for root, dirs, files in os.walk(base):

    if len(files) > 0:
        print(root, " ---> ", len(files), "images")

Preparing chest xray model

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model , Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D, GlobalMaxPool2D

In [ ]:
conv_base = DenseNet121(
    weights = 'imagenet',
    include_top = False,
    input_shape = (224,224,3)
)

In [ ]:
for i, layer in enumerate(conv_base.layers):
    print(i, layer.name)

In [ ]:
conv_base.trainable = True

for layer in conv_base.layers:
    layer.trainable = False

for layer in conv_base.layers[-50:]:
    layer.trainable = True

for layer in conv_base.layers:
    print(layer.name, layer.trainable)

In [ ]:
model = Sequential()
model.add(conv_base)
model.add(GlobalAveragePooling2D())
model.add(Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)))
model.add(Dropout(0.6)) # Increased dropout rate
model.add(Dense(1, activation='sigmoid', kernel_regularizer=keras.regularizers.l2(0.001)))

In [ ]:
conv_base.summary()

Data augumentaion

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator , load_img , img_to_array, array_to_img

In [ ]:
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale = 1./255,
    shear_range = 0.5,
    zoom_range = 0.5,
    horizontal_flip = True,
    vertical_flip = True
)

test_datagen = ImageDataGenerator(
    rescale = 1./255
)

train_generator = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/medical_project/chest_xray/chest_xray/train',
    target_size = (224,224),
    batch_size = batch_size,
    class_mode = 'binary'
)

validate_generator = test_datagen.flow_from_directory(
    '/content/drive/MyDrive/medical_project/chest_xray/chest_xray/test',
    target_size = (224,224),
    batch_size = batch_size,
    class_mode = 'binary'
)

In [ ]:
import os

# List the contents of the unzipped chest_xray directory to find the correct path
print(os.listdir('/content/drive/MyDrive/medical_project/chest_xray'))

In [ ]:
import os

# List the contents of the nested chest_xray directory
print(os.listdir('/content/drive/MyDrive/medical_project/chest_xray/chest_xray'))

In [ ]:
model.compile(optimizer= 'adam', loss = 'binary_crossentropy', metrics = ['accuracy', 'precision', 'recall'])

In [ ]:
# Earlystopping
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
     monitor="val_loss",
    min_delta=0.00001,
    patience=5,
    verbose=1,
    mode="auto",
    baseline=None,
    restore_best_weights=False
)

In [ ]:
history = model.fit(
    train_generator,
    epochs = 20,
    validation_data = validate_generator,
    callbacks = [early_stopping]
)

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.grid(True)
plt.show()

# Plot training & validation loss values
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.grid(True)
plt.show()

In [ ]:
model.compile(optimizer= 'adam', loss = 'binary_crossentropy', metrics = ['accuracy', 'precision', 'recall'])
test_loss, test_acc, test_precision, test_recall = model.evaluate(validate_generator)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss:     {test_loss:.4f}")

Testing model pred

In [ ]:
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.densenet import preprocess_input
import numpy as np

In [ ]:
img_path = '/content/drive/MyDrive/medical_project/chest_xray/chest_xray/test/NORMAL/IM-0007-0001.jpeg'

In [ ]:
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

In [ ]:
prediction = model.predict(img_array)
print(prediction)

In [ ]:
import matplotlib.pyplot as plt

# Assuming the model outputs the probability of PNEUMONIA (class index 1).
# And assuming 'NORMAL' is class index 0 and 'PNEUMONIA' is class index 1 (alphabetical order).
class_labels = ['NORMAL', 'PNEUMONIA']

# Interpret the prediction (e.g., [[0.6155932]])
pneumonia_probability = prediction[0][0]

if pneumonia_probability > 0.5:
    corrected_predicted_class = class_labels[1] # PNEUMONIA
else:
    corrected_predicted_class = class_labels[0] # NORMAL

plt.imshow(img)
plt.title(f"Prediction: {corrected_predicted_class}")
plt.axis("off")

Preparing brain mri model

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense,GlobalAveragePooling2D, Dropout
from keras.applications import EfficientNetV2B3

Load data

In [ ]:
!cp -r '/content/drive/MyDrive/medical_project/brain_mri' '/content/brain_mri'

In [ ]:
!cp -r '/content/drive/MyDrive/medical_project/brain_mri/Training' '/content/brain_mri/Training'

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import image_dataset_from_directory

train_datagen = ImageDataGenerator(
    preprocessing_function = tf.keras.applications.efficientnet_v2.preprocess_input,
    # rescale = 1./255,
    shear_range = 0.5,
    zoom_range = 0.5,
    horizontal_flip = True,
    vertical_flip = True
)

test_datagen = ImageDataGenerator(
    preprocessing_function = tf.keras.applications.efficientnet_v2.preprocess_input,
    # rescale = 1./255
)
train_generator = train_datagen.flow_from_directory(
    '/content/brain_mri//Training',
    target_size = (300,300),
    batch_size = 16,
    class_mode = 'categorical',

)
validate_generator = test_datagen.flow_from_directory(
    '/content//brain_mri/Testing',
    target_size = (300,300),
    batch_size = 16,
    class_mode = 'categorical',


)

In [ ]:
# Brain tumor dataset is imbalanced — meningioma has fewer samples
import numpy as np

class_counts = np.bincount(train_generator.classes)
for label, count in zip(train_generator.class_indices.keys(), class_counts):
    print(f"{label:15s}: {count} images")

In [ ]:
# To check class imbalances
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)

Train

In [ ]:
conv_base = EfficientNetV2B3(
    weights = 'imagenet',
    include_top= False,
    input_shape = (300,300,3))

In [ ]:
# Update img size
target_size = (300,300)

In [ ]:
conv_base.trainable = True

In [ ]:
from tensorflow.keras.models import Sequential

brain_mri_model = Sequential()
brain_mri_model.add(conv_base)
brain_mri_model.add(GlobalAveragePooling2D())
brain_mri_model.add(Dense(256, activation='relu' ))
brain_mri_model.add(Dropout(0.5))
brain_mri_model.add(Dense(4, activation='softmax'))

In [ ]:
brain_mri_model.summary()

In [ ]:
print("Trainable weights:", len(brain_mri_model.trainable_weights))
print("Non-trainable weights:", len(brain_mri_model.non_trainable_weights))

Training

In [ ]:
from tensorflow.keras.metrics import AUC
brain_mri_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-4),
    loss = 'categorical_crossentropy',
    metrics = ['accuracy', AUC(name='auc')]
    # run_eagerly=True
)

In [ ]:
# Earlystopping
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
     monitor="val_loss",
     patience = 7,
    #  min_delta = 0.00001,
    #  verbose = 1,
    #  mode = "auto",
    #  baseline = None,
     restore_best_weights = True
)

In [ ]:
import tensorflow as tf

# Convert generators to tf.dataset and cache in RAM
train_dataset = tf.data.Dataset.from_generator(
    lambda: train_generator,
    output_signature=(
        tf.TensorSpec(shape=(None, 300, 300, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, 4), dtype=tf.float32)
    )
).cache()  # caches entire dataset in RAM after first epoch

validate_dataset = tf.data.Dataset.from_generator(
    lambda: validate_generator,
    output_signature=(
        tf.TensorSpec(shape=(None, 300, 300, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, 4), dtype=tf.float32)
    )
).cache()

In [ ]:
history = brain_mri_model.fit(train_generator,
          epochs=30,
          validation_data = validate_generator,
          callbacks = [early_stopping],
          class_weight=class_weight_dict
)

checking validation

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Accuracy', marker='o')
plt.plot(epochs_range, val_acc, label='Val Accuracy', marker='o')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Loss', marker='o')
plt.plot(epochs_range, val_loss, label='Val Loss', marker='o')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Print final numbers
print(f"Final Train Accuracy : {acc[-1]:.4f}")
print(f"Final Val Accuracy   : {val_acc[-1]:.4f}")
print(f"Final Train Loss     : {loss[-1]:.4f}")
print(f"Final Val Loss       : {val_loss[-1]:.4f}")
print(f"Best Val Accuracy    : {max(val_acc):.4f} at epoch {val_acc.index(max(val_acc))+1}")

In [ ]:
# Check what the model actually predicts across the whole test set
import numpy as np

brain_class_labels = ['glioma', 'meningioma', 'notumor', 'pituitary'] # Added this line

y_pred = np.argmax(brain_mri_model.predict(validate_generator), axis=1)
y_true = validate_generator.classes

# See prediction distribution
unique, counts = np.unique(y_pred, return_counts=True)
print("Prediction distribution:")
for cls, count in zip(unique, counts):
    print(f"  Class {cls} ({brain_class_labels[cls]}): {count} times")

print("\nActual distribution:")
unique_true, counts_true = np.unique(y_true, return_counts=True)
for cls, count in zip(unique_true, counts_true):
    print(f"  Class {cls} ({brain_class_labels[cls]}): {count} times")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

brain_class_labels = ['glioma', 'meningioma', 'notumor', 'pituitary']

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=brain_class_labels,
            yticklabels=brain_class_labels,
            cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print(classification_report(y_true, y_pred,
      target_names=brain_class_labels))

Testing prediction

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import efficientnet_v2
import matplotlib.pyplot as plt

brain_class_labels = ['glioma', 'meningioma', 'notumor', 'pituitary']

def predict_brain_mri(img_path, model_to_use):
    # Load and preprocess
    img = image.load_img(img_path, target_size=(300, 300))
    img_array = image.img_to_array(img)
    img_array = efficientnet_v2.preprocess_input(img_array)  # must match training
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    predictions = model_to_use.predict(img_array, verbose=0)
    predicted_idx = np.argmax(predictions)
    predicted_class = brain_class_labels[predicted_idx]
    confidence = np.max(predictions) * 100

    # Plot
    plt.imshow(image.load_img(img_path))
    plt.axis('off')
    plt.title(f"Predicted: {predicted_class}\nConfidence: {confidence:.2f}%",
              fontsize=14, fontweight='bold',
              color='green' if confidence > 70 else 'red')
    plt.show()

    # All probabilities
    print("\nAll class probabilities:")
    for label, prob in zip(brain_class_labels, predictions[0]):
        bar = '█' * int(prob * 40)
        print(f"  {label:15s}: {prob*100:.2f}%  {bar}")

    # Confidence warning
    if confidence < 70:
        print("\n Low confidence — consult a radiologist")

In [ ]:
test_images = {
    'glioma' : '/content/brain_mri/Testing/glioma/Te-gl_103.jpg',
    'meningioma' : '/content/brain_mri/Testing/meningioma/Te-me_100.jpg',
    'notumor' : '/content/brain_mri/Testing/notumor/Te-no_1.jpg',
    'pituitary' : '/content/brain_mri/Testing/pituitary/Te-pi_100.jpg'
}

for actual_class, img_path in test_images.items():
    print(f"Actual class: {actual_class}")
    predict_brain_mri(img_path, brain_mri_model)

Final check before conactenating & GRAD-CAM application

In [ ]:
# For brain model
def get_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.Model):
            for inner_layer in reversed(layer.layers):
                if len(inner_layer.output.shape) == 4:
                    return inner_layer.name
        if len(layer.output.shape) == 4:
            return layer.name

print("Brain last conv:", get_last_conv_layer(brain_mri_model))
print("Chest last conv:", get_last_conv_layer(chest_model))

In [ ]:
# Are these your variable names?
print(brain_mri_model.summary())  # brain model
print(model.summary())      # chest model

In [ ]:
brain_mri_model.save('/content/drive/MyDrive/medical_project/brain_model_final.keras')
model.save('/content/drive/MyDrive/medical_project/chest_model_final.keras')

In [ ]:
print(type(brain_mri_model))
print(type(model))

In [ ]:
print("Brain input:", brain_mri_model.input_shape)
print("Chest input:", chest_model.input_shape)

GRAD-CAM application

In [ ]:
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import efficientnet_v2
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

In [ ]:
from tensorflow.keras.models import load_model

# 3. Load both models
brain_mri_model = load_model('/content/drive/MyDrive/medical_project/brain_model_final.keras')
chest_model = load_model('/content/drive/MyDrive/medical_project/chest_model_final.keras')
print("Both loaded")

In [ ]:
# Copy Testing data
!cp -r '/content/drive/MyDrive/medical_project/brain_mri/Testing' '/content/brain_mri/Testing'

In [ ]:
!ls '/content/brain_mri/Testing'

In [ ]:
!cp -r '/content/drive/MyDrive/medical_project/chest_xray' '/content/chest_xray'

In [ ]:
!ls '/content/chest_xray/test'

In [ ]:
 # GRAD-CAM core functions
import tensorflow as tf
import numpy as np
import cv2

def get_gradcam_heatmap(model, img_array, last_conv_layer_name):
    # Ensure img_array is a Tensor
    img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)

    # 1. Create a model that outputs the activations of the last convolutional layer
    # The base_conv_model is the first layer of the Sequential model (e.g., EfficientNetV2B3).
    base_conv_model = model.layers[0]

    # Create the feature extraction model: input to last_conv_layer_name
    feature_extractor_model = tf.keras.Model(
        inputs=base_conv_model.input, # Use the input of the base conv model
        outputs=base_conv_model.get_layer(last_conv_layer_name).output
    )

    # 2. Create the classifier model (from the output of last_conv_layer to the final prediction)
    # The input shape for this classifier head will be the output shape of the last_conv_layer.
    classifier_input_shape = feature_extractor_model.output_shape[1:]
    classifier_input = tf.keras.Input(shape=classifier_input_shape)

    # Build the classifier head by taking layers from the original Sequential model
    # that come *after* the base_conv_model.
    x = classifier_input
    for layer in model.layers[1:]: # Iterate through layers after the base_conv_model
        x = layer(x)
    classifier_model = tf.keras.Model(inputs=classifier_input, outputs=x)

    with tf.GradientTape() as tape:
        # Pass the input image through the feature extractor to get convolutional features
        # And ensure these features are watched by the tape
        conv_features = feature_extractor_model(img_tensor)
        tape.watch(conv_features)

        # Pass the convolutional features through the classifier to get predictions
        predictions = classifier_model(conv_features)

        # Get the score of the predicted class
        predicted_class_idx = tf.argmax(predictions[0])
        class_score = predictions[:, predicted_class_idx]

    # Compute gradients of the class score with respect to the convolutional features
    grads = tape.gradient(class_score, conv_features)

    # Perform global average pooling on the gradients and multiply with feature maps to create heatmap.
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_features[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize heatmap to [0, 1] for visualization.
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)

    # Also return the original model's predictions for display
    full_model_predictions = model.predict(img_tensor, verbose=0)

    return heatmap.numpy(), full_model_predictions

def overlay_gradcam(img_path, heatmap, target_size):
    img = cv2.imread(img_path)
    img = cv2.resize(img, target_size)
    heatmap_resized = cv2.resize(heatmap, target_size)
    heatmap_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
    )
    superimposed = cv2.addWeighted(img, 0.6, heatmap_colored, 0.4, 0)
    superimposed = cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)
    return superimposed

Model Configration

In [ ]:
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import efficientnet_v2
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

In [ ]:
MODEL_CONFIG = {
    'brain': {
        'model'           : brain_mri_model,
        'class_labels'    : ['glioma', 'meningioma', 'notumor', 'pituitary'],
        'preprocess_fn'   : efficientnet_v2.preprocess_input,
        'target_size'     : (300, 300),
        'last_conv_layer' : 'top_activation',
        'title'           : 'Brain Tumor MRI Classification'
    },
    'chest': {
        'model'           : chest_model,
        'class_labels'    : ['NORMAL', 'PNEUMONIA'],
        'preprocess_fn'   : densenet_preprocess,
        'target_size'     : (224, 224),
        'last_conv_layer' : 'relu',
        'binary'          : True,
        'title'           : 'Chest X-Ray Classification'
    }
}

In [ ]:
def predict_with_gradcam(img_path, model, class_labels,
                          last_conv_layer_name, preprocess_fn,
                          target_size, modality, binary=False):

    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = preprocess_fn(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    heatmap, predictions = get_gradcam_heatmap(
        model, img_array, last_conv_layer_name
    )

    # Ensure predictions is a numpy array for consistent indexing and operations
    predictions = np.asarray(predictions)

    # ── Handle binary sigmoid output ──
    if binary:
        prob_pneumonia = float(predictions[0][0])
        prob_normal = 1 - prob_pneumonia
        probs = [prob_normal, prob_pneumonia]
        predicted_idx = 1 if prob_pneumonia > 0.5 else 0
        confidence = max(probs) * 100
    else:
        probs = predictions[0]
        predicted_idx = int(np.argmax(probs))
        confidence = float(np.max(probs)) * 100

    predicted_class = class_labels[predicted_idx]

    # ── Plot ──
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(image.load_img(img_path))
    axes[0].set_title('Original')
    axes[0].axis('off')

    if heatmap is not None:
        superimposed = overlay_gradcam(img_path, heatmap, target_size)
        axes[1].imshow(superimposed)
        axes[1].set_title('Grad-CAM')
        axes[1].axis('off')

    axes[2].barh(class_labels, [p*100 for p in probs], color='steelblue')
    axes[2].set_xlim(0, 100)
    axes[2].set_xlabel('Confidence %')
    axes[2].set_title('Probabilities')
    plt.tight_layout()
    plt.show()

    print(f"\n{'='*40}")
    print(f"Modality  : {modality.upper()}")
    print(f"Predicted : {predicted_class}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"{'='*40}")
    print("\nAll class probabilities:")
    for label, prob in zip(class_labels, probs):
        bar = '█' * int(prob * 30)
        print(f"  {label:15s}: {prob*100:.2f}%  {bar}")

    if confidence < 70:
        print("\n⚠️  Low confidence — consult a radiologist")

    return predicted_class, confidence

In [ ]:
# Brain MRI — one image from each class
brain_test_images = {
    'glioma'     : '/content/brain_mri/Testing/glioma/Te-gl_155.jpg',
    'meningioma' : '/content/brain_mri/Testing/meningioma/Te-me_224.jpg',
    'notumor'    : '/content/brain_mri/Testing/notumor/Te-no_93.jpg',
    'pituitary'  : '/content/brain_mri/Testing/pituitary/Te-pi_367.jpg',
}

for actual_class, path in brain_test_images.items():
    print(f"\nActual: {actual_class.upper()}")
    config = MODEL_CONFIG['brain']
    predict_with_gradcam(
        path,
        model=config['model'],
        class_labels=config['class_labels'],
        last_conv_layer_name=config['last_conv_layer'],
        preprocess_fn=config['preprocess_fn'],
        target_size=config['target_size'],
        modality='brain',
        binary=config.get('binary', False)
    )

# Chest X-Ray — one image from each class
chest_test_images = {
    'NORMAL'    : '/content/chest_xray/chest_xray/chest_xray/test/NORMAL/IM-0001-0001.jpeg',
    'PNEUMONIA' : '/content/chest_xray/chest_xray/chest_xray/test/PNEUMONIA/person101_bacteria_486.jpeg',
}

for actual_class, path in chest_test_images.items():
    print(f"\nActual: {actual_class.upper()}")
    config = MODEL_CONFIG['chest']
    predict_with_gradcam(
        path,
        model=config['model'],
        class_labels=config['class_labels'],
        last_conv_layer_name=config['last_conv_layer'],
        preprocess_fn=config['preprocess_fn'],
        target_size=config['target_size'],
        modality='chest',
        binary=config.get('binary', False)
    )


In [ ]:
print(chest_model.layers[-1].get_config())

In [ ]:
brain_mri_model.save('/content/drive/MyDrive/medical_project/brain_model_final.keras')
chest_model.save('/content/drive/MyDrive/medical_project/chest_model_final.keras')
print("Both saved ")